In [1]:
import sys
import os

# Use current working directory and go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you can import your config
from config import api_key

In [2]:
import json

function_definition = [
    {'type': 'function',
     'function': {'name': 'extract_structured_data',
     'description': 'extract dates, descriptions and amounts from text and return in in a',
     'parameters': {'type': 'object',
     'properties': {'home type': {'type': 'string', 'description': 'Home type'},
     'location': {'type': 'string', 'description': 'Location'},
     'price': {'type': 'integer', 'description': 'Price'},
     'bedrooms': {'type': 'integer', 'description': 'Number of bedrooms'}}}}}
]
print(json.dumps(function_definition,indent=4))

[
    {
        "type": "function",
        "function": {
            "name": "extract_structured_data",
            "description": "extract dates, descriptions and amounts from text and return in in a",
            "parameters": {
                "type": "object",
                "properties": {
                    "home type": {
                        "type": "string",
                        "description": "Home type"
                    },
                    "location": {
                        "type": "string",
                        "description": "Location"
                    },
                    "price": {
                        "type": "integer",
                        "description": "Price"
                    },
                    "bedrooms": {
                        "type": "integer",
                        "description": "Number of bedrooms"
                    }
                }
            }
        }
    }
]


In [3]:
import pdfplumber

# PDF lezen
with pdfplumber.open("AFSCHRIFT.pdf") as pdf:
    text = "\n".join(page.extract_text() for page in pdf.pages)

Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported
Cannot set non-stroke color because 2 components are specified but only 1 (grayscale), 3 (rgb) and 4 (cmyk) are supported


In [4]:
text

'Afschrift Platinumcard\nOp ING.nl vind je de antwoorden op veel van je vragen.\nWil je persoonlijk contact? Kijk dan op ing.nl/contact\nPeriode\n05-07-2025 t/m 04-08-2025\nHr S van Weeren\nJohan Willem Frisolaan 27 Rekeningnummer\nNL93 INGB 0005 7762 48\n2252 HC VOORSCHOTEN\nOvereenkomstnummer\n2100 1259 7824\nKredietlimiet (EUR) Bestedingsruimte (EUR)\n7.000,00 7.000,00\nOp 06-08-2025 schrijven wij 900,91 euro af van uw betaalrekening met nummer NL93 INGB 0005 7762 48.\nGeboekt op Naam / Omschrijving / Mededeling Type Bedrag (EUR)\n04-08-2025 AFLOSSING Incasso +900,91\nKaartnummer: 5248 **** **** 5478\n03-08-2025 Google Play Apps Dublin Betaling -11,99\nTransactiedatum: 02-08-2025\nKaartnummer: 5248 **** **** 5478\n03-08-2025 MERPAGO*MATAMBRE CIUDAD DE MEX Betaling -28,33\nTransactiedatum: 02-08-2025\nKaartnummer: 5248 **** **** 5478\nBedrag: 593,25 MXN\nKoers: 0,04682\nKoersopslag (EUR): 0,56\n03-08-2025 MERPAGO*GISELAANCONA CIUDAD DE MEX Betaling -120,35\nTransactiedatum: 02-08-202

In [21]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

# Naar OpenAI sturen voor structuur
response = client.responses.create(
    model="gpt-4.1",  # of gpt-4o
    input=f"Extract the dates (Geboekt op), the description (Naam / Omschrijving / Mededeling), the amounts (Bedrag (EUR)) and return only a response in JSON (I do not want to see the word json in the respone) format from the text provided in triple quotes ```{text}```"
)

print(response.output_text)

[
  {
    "date": "04-08-2025",
    "description": "AFLOSSING Incasso",
    "amount": "+900,91"
  },
  {
    "date": "03-08-2025",
    "description": "Google Play Apps Dublin",
    "amount": "-11,99"
  },
  {
    "date": "03-08-2025",
    "description": "MERPAGO*MATAMBRE CIUDAD DE MEX",
    "amount": "-28,33"
  },
  {
    "date": "03-08-2025",
    "description": "MERPAGO*GISELAANCONA CIUDAD DE MEX",
    "amount": "-120,35"
  },
  {
    "date": "02-08-2025",
    "description": "HOTEL HACIENDA CHICHEN TINUM YUC",
    "amount": "-310,87"
  },
  {
    "date": "02-08-2025",
    "description": "AWS EMEA aws.amazon.co",
    "amount": "-0,83"
  },
  {
    "date": "02-08-2025",
    "description": "REST BARQUITO MAWIMBI LAZARO CARDEN",
    "amount": "-82,93"
  },
  {
    "date": "01-08-2025",
    "description": "MERPAGO*ELPESCADOR CIUDAD DE MEX",
    "amount": "-12,62"
  },
  {
    "date": "01-08-2025",
    "description": "MERPAGO*GLENDYTOURS CIUDAD DE MEX",
    "amount": "-255,48"
  },
  {
    

In [45]:
import json

#print(response.output_text)
print(type(response.output_text))

json_data = json.loads(response.output_text)
type(json_data)

<class 'str'>


list

In [46]:
import pandas as pd
df = pd.DataFrame(json_data)
df

,date,description,amount
0,04-08-2025,AFLOSSING Incasso,"+900,91"
1,03-08-2025,Google Play Apps Dublin,"-11,99"
2,03-08-2025,MERPAGO*MATAMBRE CIUDAD DE MEX,"-28,33"
3,03-08-2025,MERPAGO*GISELAANCONA CIUDAD DE MEX,"-120,35"
4,02-08-2025,HOTEL HACIENDA CHICHEN TINUM YUC,"-310,87"
5,02-08-2025,AWS EMEA aws.amazon.co,"-0,83"
6,02-08-2025,REST BARQUITO MAWIMBI LAZARO CARDEN,"-82,93"
7,01-08-2025,MERPAGO*ELPESCADOR CIUDAD DE MEX,"-12,62"
8,01-08-2025,MERPAGO*GLENDYTOURS CIUDAD DE MEX,"-255,48"
9,01-08-2025,Betaling naar creditcard Correctie,"+800,00"


In [47]:
df["amount"] = df["amount"].str.replace(".", "", regex=False).str.replace(",", ".", regex=False).astype(float)

In [48]:
df

,date,description,amount
0,04-08-2025,AFLOSSING Incasso,900.91
1,03-08-2025,Google Play Apps Dublin,-11.99
2,03-08-2025,MERPAGO*MATAMBRE CIUDAD DE MEX,-28.33
3,03-08-2025,MERPAGO*GISELAANCONA CIUDAD DE MEX,-120.35
4,02-08-2025,HOTEL HACIENDA CHICHEN TINUM YUC,-310.87
5,02-08-2025,AWS EMEA aws.amazon.co,-0.83
6,02-08-2025,REST BARQUITO MAWIMBI LAZARO CARDEN,-82.93
7,01-08-2025,MERPAGO*ELPESCADOR CIUDAD DE MEX,-12.62
8,01-08-2025,MERPAGO*GLENDYTOURS CIUDAD DE MEX,-255.48
9,01-08-2025,Betaling naar creditcard Correctie,800.00


In [50]:
df.amount.sum()

np.float64(-4.796163466380676e-14)